In [ ]:
import cv2
import numpy as np
import os

# =========================================================
# CONFIGURACIÓN
# =========================================================

WIDTH, HEIGHT = 600, 600

# Cargar imagen
input_path = "imagenes/img_9.png"

img = cv2.imread(input_path)

if img is None:
    print("No se pudo cargar la imagen.")
    exit()

h, w = img.shape[:2]

# =========================================================
# 1. CREAR ROI TRAPEZOIDAL DEL SUELO
# =========================================================

# Trapecio aproximando el suelo del pasillo --> Cuando tengamos la camara funcionando habrá que ajustar quizas

pts_roi = np.array([
    [0, h],                    # esquina inferior izquierda
    [w, h],                    # esquina inferior derecha
    [int(0.65 * w), int(0.55 * h)],  # superior derecha
    [int(0.35 * w), int(0.55 * h)]   # superior izquierda
], dtype=np.int32)

# Crear máscara negra
mask = np.zeros((h, w), dtype=np.uint8)

# Dibujar el trapecio blanco
cv2.fillConvexPoly(mask, pts_roi, 255)

# =========================================================
# 2. ORB SOLO EN EL SUELO
# =========================================================

orb = cv2.ORB_create(
    nfeatures=1000,
    fastThreshold=5
)

# Detectar SOLO dentro de la máscara
keypoints, descriptors = orb.detectAndCompute(img, mask)

# Dibujar keypoints
img_puntos = cv2.drawKeypoints(
    img,
    keypoints,
    None,
    color=(0, 255, 0)
)

# Dibujar ROI para visualizar
cv2.polylines(
    img_puntos,
    [pts_roi],
    isClosed=True,
    color=(255, 0, 0),
    thickness=2
)

# =========================================================
# 3. DEFINIR PUNTOS PARA HOMOGRAFÍA
# =========================================================

# Los 4 puntos del trapecio del suelo
pts_src = np.array([
    [0, h],
    [w, h],
    [int(0.65 * w), int(0.55 * h)],
    [int(0.35 * w), int(0.55 * h)]
], dtype=np.float32)

# Vista cenital destino
pts_dst = np.array([
    [0, HEIGHT],
    [WIDTH, HEIGHT],
    [WIDTH, 0],
    [0, 0]
], dtype=np.float32)

# =========================================================
# 4. CALCULAR HOMOGRAFÍA
# =========================================================

# Calcula el movimiento
H = cv2.getPerspectiveTransform(pts_src, pts_dst)

# Genera el movimiento
top_down = cv2.warpPerspective(img,H,(WIDTH, HEIGHT))

# =========================================================
# 5. VISUALIZACIÓN
# =========================================================

# Dibujar puntos usados
img_check = img.copy()

for i, pt in enumerate(pts_src):
    cv2.circle(
        img_check,
        tuple(pt.astype(int)),
        7,
        (0, 255, 0),
        -1
    )

    cv2.putText(
        img_check,
        str(i + 1),
        tuple(pt.astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

# Dibujar trapecio
cv2.polylines(
    img_check,
    [pts_src.astype(np.int32)],
    True,
    (255, 0, 0),
    2
)

# =========================================================
# 6. MOSTRAR RESULTADOS
# =========================================================

cv2.imshow("Mascara ROI", mask)
cv2.imshow("Puntos ORB SOLO SUELO", img_puntos)
cv2.imshow("Puntos Usados Homografia", img_check)
cv2.imshow("Vista Cenital", top_down)

cv2.waitKey(0)
cv2.destroyAllWindows()

# =========================================================
# GUARDAR IMAGEN CENITAL
# =========================================================

# Crear carpeta si no existe
os.makedirs("imagenesCenitales", exist_ok=True)

# Obtener nombre del archivo sin extensión
nombre = os.path.splitext(os.path.basename(input_path))[0]

# Obtener extensión original
extension = os.path.splitext(input_path)[1]

# Crear nuevo nombre
output_path = f"imagenesCenitales/{nombre}_cenital{extension}"

# Guardar imagen
cv2.imwrite(output_path, top_down)

Homografía realizada usando ROI trapezoidal.
Imagen guardada en: imagenesCenitales/img_9_cenital.png
